<a href="https://colab.research.google.com/github/pronabpaul/Linear-Algebra/blob/main/PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Is the Dataset High-Dimensional?


A digital image consists of a grid of pixel intensities. To analyze a collection of images mathematically, each image is reshaped into a one-dimensional vector.

The dataset is represented as a matrix

$$
X \in \mathbb{R}^{n \times d},
$$

where:

- $n$ is the number of images.
- $d$ is the number of pixels in each image.

For example, **500** grayscale images with a resolution of **256 x 256** pixels form the matrix

$$
X \in \mathbb{R}^{500 \times 65536}.
$$



## Interpretation

Each image is represented as a point in a **65,536-dimensional** feature space.

Such a high-dimensional representation is difficult to visualize, interpret, and analyze directly because each image contains tens of thousands of pixel values.


## Insight

Dimensionality reduction techniques such as **PCA** and **SVD** transform the original high-dimensional data into a lower-dimensional representation that preserves the most important information.

This lower-dimensional space makes it possible to visualize image distributions, identify dominant patterns, and perform tasks such as compression, denoising, clustering, and classification more effectively.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
from sklearn.decomposition import PCA

def load_images(paths, size=128):
    images = []
    for p in paths:
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            img = cv2.resize(img, (size, size))
            images.append(img.flatten().astype(np.float64))
    if not images:
        raise ValueError("No valid images.")
    return np.stack(images)

def main():
    paths = [
        "/content/test.webp",
        "/content/test2.jpg",
        "/content/test3.jpg",
        "/content/test_4.jpeg"
    ]
    X = load_images(paths, size=128)
    n, d = X.shape
    h = w = 128

    print(f"X: {n}×{d}")

    # Center and PCA
    Xc = X - X.mean(axis=0)
    pca = PCA()
    pca.fit(Xc)
    cumsum = np.cumsum(pca.explained_variance_ratio_)
    k_95 = np.argmax(cumsum >= 0.95) + 1

    # Original images
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    if n == 1:
        axes = [axes]
    for i, ax in enumerate(axes):
        ax.imshow(X[i].reshape(h, w), cmap='gray')
        ax.set_title(f'Image {i+1}')
        ax.axis('off')
    plt.suptitle('Original Images')
    plt.tight_layout()
    plt.show()

    # Mean image
    mean_img = X.mean(axis=0).reshape(h, w)
    plt.figure(figsize=(4, 4))
    plt.imshow(mean_img, cmap='gray')
    plt.title('Mean Image')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    # Top 3 eigenimages
    components = pca.components_[:3]
    fig, axes = plt.subplots(1, 3, figsize=(9, 3))
    for i, ax in enumerate(axes):
        eigen = components[i].reshape(h, w)
        vmax = np.max(np.abs(eigen))
        ax.imshow(eigen, cmap='RdBu', vmin=-vmax, vmax=vmax)
        ax.set_title(f'PC{i+1}')
        ax.axis('off')
    plt.suptitle('Principal Components (Eigenimages)')
    plt.tight_layout()
    plt.show()

    # Cumulative explained variance
    plt.figure(figsize=(6, 4))
    plt.plot(np.arange(1, len(cumsum)+1), cumsum, 'o-', color='black', linewidth=2)
    plt.axhline(y=0.95, color='red', linestyle='--', linewidth=1.5, label='95%')
    plt.axvline(x=k_95, color='red', linestyle='--', linewidth=1.5, alpha=0.5)
    plt.xlabel('Principal components')
    plt.ylabel('Cumulative explained variance')
    plt.title('Dimensionality reduction')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    # 2D projection (first two PCs)
    Z = pca.transform(Xc)[:, :2]
    plt.figure(figsize=(8, 6))
    plt.scatter(Z[:, 0], Z[:, 1], s=80, edgecolors='black')
    for i in range(n):
        plt.annotate(str(i+1), (Z[i,0], Z[i,1]), fontsize=10, ha='center', va='bottom')
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
    plt.title('Images projected onto first 2 PCs')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"Original dimensions: {d}")
    print(f"Reduced dimensions (95% variance): {k_95}")
    print(f"Compression: {d} → {k_95} ({100*(1 - k_95/d):.1f}% reduction)")

if __name__ == "__main__":
    main()

# How Many Dimensions Do We Actually Need?


Although each image is represented by thousands of pixel values, much of the information is often redundant. **Principal Component Analysis (PCA)** identifies a smaller set of orthogonal components that capture the dominant variations in the dataset.

The explained variance ratio measures how much of the total dataset variance is captured by each principal component. The cumulative explained variance is computed as

$$
C(k)=\sum_{i=1}^{k}\text{Explained Variance Ratio}_i,
$$

where $C(k)$ represents the proportion of total variance preserved by the first $k$ principal components.



## Interpretation

The cumulative explained variance curve shows how much information is retained as additional principal components are included.

Common thresholds used in practice are:

- **90%** - Preserves most of the important information with substantial dimensionality reduction.
- **95%** - A widely used balance between model simplicity and information retention.
- **99%** - Retains nearly all of the original information while requiring more components.



## Insight

The smallest number of principal components that achieves a desired variance threshold represents the effective dimensionality of the dataset.

This analysis is often the first step in medical image analysis because it helps determine how much the original feature space can be reduced while preserving the essential information needed for visualization, classification, and other downstream tasks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
from sklearn.decomposition import PCA

def load_images(paths, size=128):
    images = []
    for p in paths:
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            img = cv2.resize(img, (size, size))
            images.append(img.flatten().astype(np.float64))
    if not images:
        raise ValueError("No valid images.")
    return np.stack(images)

def main():
    paths = [
        "/content/test.webp",
        "/content/test2.jpg",
        "/content/test3.jpg",
        "/content/test_4.jpeg"
    ]
    X = load_images(paths, size=128)
    n, d = X.shape
    print(f"Dataset: {n} images × {d} pixels")

    Xc = X - X.mean(axis=0)
    pca = PCA()
    pca.fit(Xc)

    cumsum = np.cumsum(pca.explained_variance_ratio_)

    k90 = np.argmax(cumsum >= 0.90) + 1
    k95 = np.argmax(cumsum >= 0.95) + 1
    k99 = np.argmax(cumsum >= 0.99) + 1

    print(f"90% variance: {k90} components")
    print(f"95% variance: {k95} components")
    print(f"99% variance: {k99} components")
    print(f"Reduction: {d} → {k95} ({100*(1-k95/d):.1f}% compression)")

    plt.figure(figsize=(6, 4))
    plt.plot(np.arange(1, len(cumsum)+1), cumsum, 'o-', color='black', linewidth=2)
    plt.axhline(y=0.90, color='blue', linestyle='--', label='90%')
    plt.axhline(y=0.95, color='red', linestyle='--', label='95%')
    plt.axhline(y=0.99, color='green', linestyle='--', label='99%')
    plt.xlabel('Principal components')
    plt.ylabel('Cumulative explained variance')
    plt.title('PCA dimensionality reduction')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    main()

# Visualizing Images in Two Dimensions


Principal Component Analysis (PCA) projects high-dimensional image data into a lower-dimensional space while preserving as much of the original variation as possible.

Given the image matrix

$$
X \in \mathbb{R}^{n \times d},
$$

PCA computes a two-dimensional representation

$$
Z \in \mathbb{R}^{n \times 2},
$$

where each row of $Z$ represents one image projected onto the first two principal components.


## Interpretation

Each point in the projection corresponds to a single image.

Images with similar visual characteristics tend to cluster together, whereas images with different characteristics are often located farther apart in the projection space.

This visualization can reveal:

- Natural clusters within the dataset.
- Relationships between different image classes.
- Outliers or unusual image samples.


## Insight

Although each image may contain thousands of pixel values, the first two principal components often capture the dominant sources of variation in the dataset.

Visualizing the data in this two-dimensional space provides an intuitive way to explore its underlying structure before applying machine learning or deep learning models.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
from sklearn.decomposition import PCA

def load_images(paths, size=128):
    images = []
    for p in paths:
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            img = cv2.resize(img, (size, size))
            images.append(img.flatten().astype(np.float64))
    if not images:
        raise ValueError("No valid images.")
    return np.stack(images)

def main():
    paths = [
        "/content/test.webp",
        "/content/test2.jpg",
        "/content/test3.jpg",
        "/content/test_4.jpeg",
        "/content/xray (1).jpg"
    ]
    labels = ["MRI", "MRI", "MRI", "MRI", "X-ray"]

    X = load_images(paths, size=128)
    n, d = X.shape
    print(f"Dataset: {n} images × {d} pixels")

    Xc = X - X.mean(axis=0)
    pca = PCA(n_components=2)
    Z = pca.fit_transform(Xc)

    unique_labels = np.unique(labels)
    colors = {"MRI": "blue", "X-ray": "red"}

    plt.figure(figsize=(8, 6))
    for label in unique_labels:
        idx = np.where(np.array(labels) == label)[0]
        plt.scatter(Z[idx, 0], Z[idx, 1], c=colors[label], label=label, s=80, edgecolors='black')
    for i in range(n):
        plt.annotate(str(i+1), (Z[i,0], Z[i,1]), fontsize=9, ha='center', va='bottom')
    plt.axhline(0, color='gray', linewidth=0.8)
    plt.axvline(0, color='gray', linewidth=0.8)
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
    plt.title('2D PCA projection')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"PC1 explains {pca.explained_variance_ratio_[0]:.2%} of variance")
    print(f"PC2 explains {pca.explained_variance_ratio_[1]:.2%} of variance")
    print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.2%}")

if __name__ == "__main__":
    main()

# Detecting Outliers with PCA

PCA projects high-dimensional images into a low-dimensional space while preserving the dominant variations in the dataset.

After projection, the distance of each image from the center of the PCA space can be computed. Images that lie far from the center are considered potential outliers because they differ substantially from the majority of the dataset.


## Interpretation

Images with large distances from the center may indicate:

- Corrupted or low-quality images.
- Unusual anatomical structures.
- Rare disease patterns.
- Preprocessing or acquisition artifacts.
- Scanner or imaging protocol differences.

These samples should be examined carefully before further analysis.


## Insight

Outlier detection in PCA space provides a simple and effective way to identify images that deviate from the overall distribution of the dataset.

This technique is widely used as an initial quality-control step in medical imaging pipelines to detect abnormal or problematic samples before model training.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
from sklearn.decomposition import PCA
from scipy.spatial.distance import cdist

def load_images(paths, size=128):
    images = []
    for p in paths:
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            img = cv2.resize(img, (size, size))
            images.append(img.flatten().astype(np.float64))
    if not images:
        raise ValueError("No valid images.")
    return np.stack(images)

def main():
    paths = [
        "/content/test.webp",
        "/content/test2.jpg",
        "/content/test3.jpg",
        "/content/test_4.jpeg"
    ]
    X = load_images(paths, size=128)
    n, d = X.shape
    h = w = 128

    # PCA
    Xc = X - X.mean(axis=0)
    pca = PCA(n_components=2)
    Z = pca.fit_transform(Xc)

    center = Z.mean(axis=0)
    distances = cdist(Z, [center]).flatten()

    outlier_idx = np.argmax(distances)
    closest_idx = np.argmin(distances)

    print("Distances from PCA center (ranked):")
    ranked = np.argsort(distances)[::-1]
    for rank, idx in enumerate(ranked, 1):
        print(f"  {rank}. Image {idx+1}: {distances[idx]:.4f}")
    print(f"\nPotential outlier: Image {outlier_idx+1} (distance = {distances[outlier_idx]:.4f})")
    print(f"Closest to center: Image {closest_idx+1} (distance = {distances[closest_idx]:.4f})")

    # ---- All images ----
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    for i, ax in enumerate(axes):
        ax.imshow(X[i].reshape(h, w), cmap='gray')
        ax.set_title(f"Img {i+1}\n{distances[i]:.1f}")
        ax.axis('off')
        if i == outlier_idx:
            for spine in ax.spines.values():
                spine.set_edgecolor('red')
                spine.set_linewidth(4)
        elif i == closest_idx:
            for spine in ax.spines.values():
                spine.set_edgecolor('green')
                spine.set_linewidth(4)
    plt.suptitle('Images: Red = Outlier, Green = Closest to center')
    plt.tight_layout()
    plt.show()

    # ---- PCA scatter with outlier ----
    plt.figure(figsize=(8, 6))

    for point in Z:
        plt.plot([center[0], point[0]], [center[1], point[1]], color='gray', alpha=0.15)

    plt.scatter(Z[:, 0], Z[:, 1], s=60, edgecolors='black', zorder=3)

    plt.scatter(Z[outlier_idx, 0], Z[outlier_idx, 1],
                s=180, facecolors='none', edgecolors='red', linewidths=2,
                label='Outlier', zorder=4)

    plt.scatter(Z[closest_idx, 0], Z[closest_idx, 1],
                s=180, facecolors='none', edgecolors='green', linewidths=2,
                label='Closest', zorder=4)

    plt.scatter(center[0], center[1], c='red', marker='x', s=200, label='Center', zorder=5)
    for i in range(n):
        plt.annotate(str(i+1), (Z[i,0], Z[i,1]), fontsize=9, ha='center', va='bottom')
    plt.axhline(0, color='gray', linewidth=0.5)
    plt.axvline(0, color='gray', linewidth=0.5)
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
    plt.title('PCA Projection: Outlier (red) and Closest (green)')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    main()

# Comparing Different Image Datasets

Different image datasets often exhibit different levels of variability and complexity. By applying **PCA** to each dataset independently, we can compare their underlying structure in a quantitative way.

For each dataset, PCA is used to analyze:

- The cumulative explained variance.
- The number of principal components required to preserve a given percentage of the variance (e.g., **90%** or **95%**).
- The distribution of images in the principal component space.


## Interpretation

Comparing these characteristics provides insight into the complexity of each dataset.

- A dataset that requires more principal components to reach the same explained variance generally contains greater variability.
- A dataset that requires fewer components can often be represented more compactly with less information loss.
- Differences in the PCA projections reveal how the images are distributed in the underlying feature space.


## Insight

PCA provides a simple way to compare the intrinsic structure of different image datasets.

Such comparisons help researchers understand dataset complexity, estimate the amount of dimensionality reduction that is appropriate, and identify differences in the variability of images before developing machine learning or deep learning models.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
from sklearn.decomposition import PCA
from PIL import Image

def load_image(path, size=64):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is not None:
        img = cv2.resize(img, (size, size))
        return img.flatten().astype(np.float64)

    try:
        img = Image.open(path).convert('L')
        img = img.resize((size, size))
        return np.array(img).flatten().astype(np.float64)
    except Exception as e:
        print(f"Skipping {path}: {e}")
        return None

def load_images(paths, size=64):
    images = []
    for p in paths:
        img = load_image(p, size)
        if img is not None:
            images.append(img)
    if not images:
        raise ValueError("No images loaded.")
    return np.stack(images)

def analyze_dataset(X, name):
    Xc = X - X.mean(axis=0)
    pca = PCA()
    pca.fit(Xc)
    cumsum = np.cumsum(pca.explained_variance_ratio_)
    k90 = np.argmax(cumsum >= 0.90) + 1
    k95 = np.argmax(cumsum >= 0.95) + 1
    print(f"{name}: 90% → {k90} PCs, 95% → {k95} PCs")
    pca2 = PCA(n_components=2)
    Z = pca2.fit_transform(Xc)
    return cumsum, pca2.explained_variance_ratio_, Z

def main():
    mri_paths = [
        "/content/test.webp",
        "/content/test2.jpg",
        "/content/test3.jpg",
        "/content/test_4.jpeg"
    ]
    xray_paths = [
        "/content/Xray3.jpeg",
        "/content/Xray4.jpg",
        "/content/Xray2.avif",
        "/content/xray (1).jpg"
    ]

    mri = load_images(mri_paths, size=64)
    xray = load_images(xray_paths, size=64)
    print(f"MRI: {mri.shape[0]} images, X-ray: {xray.shape[0]} images\n")

    cumsum_mri, var_mri, Z_mri = analyze_dataset(mri, "MRI")
    cumsum_xray, var_xray, Z_xray = analyze_dataset(xray, "X-ray")

    # Cumulative explained variance
    plt.figure(figsize=(6, 4))
    plt.plot(np.arange(1, len(cumsum_mri)+1), cumsum_mri, 'o-', label='MRI', color='blue')
    plt.plot(np.arange(1, len(cumsum_xray)+1), cumsum_xray, 'o-', label='X-ray', color='red')
    plt.axhline(y=0.95, color='gray', linestyle='--', label='95%')
    plt.xlabel('Principal components')
    plt.ylabel('Cumulative explained variance')
    plt.title('Dataset complexity comparison')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    # 2D projections
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].scatter(Z_mri[:,0], Z_mri[:,1], s=40, edgecolors='black')
    axes[0].set_xlabel(f'PC1 ({var_mri[0]:.1%})')
    axes[0].set_ylabel(f'PC2 ({var_mri[1]:.1%})')
    axes[0].set_title('MRI projection')
    axes[0].grid(alpha=0.3)
    axes[1].scatter(Z_xray[:,0], Z_xray[:,1], s=40, edgecolors='black')
    axes[1].set_xlabel(f'PC1 ({var_xray[0]:.1%})')
    axes[1].set_ylabel(f'PC2 ({var_xray[1]:.1%})')
    axes[1].set_title('X-ray projection')
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    print("\nComparison summary:")
    print(f"MRI: 90% at {np.argmax(np.cumsum(cumsum_mri)>=0.90)+1} PCs, 95% at {np.argmax(np.cumsum(cumsum_mri)>=0.95)+1} PCs")
    print(f"X-ray: 90% at {np.argmax(np.cumsum(cumsum_xray)>=0.90)+1} PCs, 95% at {np.argmax(np.cumsum(cumsum_xray)>=0.95)+1} PCs")
    print("\nInterpretation: The dataset needing more PCs for the same variance is intrinsically more complex.")

if __name__ == "__main__":
    main()

# Comparing Pixel Space and Deep Feature Space

Medical images can be represented either by their raw pixel intensities or by deep feature embeddings extracted from pretrained neural networks.

PCA is applied independently to each representation to analyze how image information is distributed in the corresponding feature space.


## Comparison

| Representation | Typical Dimension | Information Captured |
|---------------|------------------:|----------------------|
| Pixel Space | 16,384+ | Raw pixel intensities |
| CNN Feature Space | 512 | Learned visual patterns |
| Vision Transformer Feature Space | 768 | Context-aware semantic representations |


## Interpretation

Each representation captures a different level of image information.

- **Pixel space** describes images using individual intensity values and typically exhibits high dimensionality.
- **CNN feature space** encodes discriminative visual patterns learned through convolutional filters.
- **Vision Transformer feature space** captures global contextual relationships and high-level semantic information.

Comparing the corresponding PCA projections reveals how the underlying image representation influences the geometry of the data.


## Insight

Deep feature embeddings provide compact representations that emphasize meaningful visual characteristics while suppressing irrelevant pixel-level variations.

Consequently, they often require fewer principal components to explain most of the variance and produce clearer separation between different image classes than representations based solely on raw pixel intensities.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch
from torchvision import transforms, models
from sklearn.decomposition import PCA
from PIL import Image

def load_image(path, size=224):
    img = cv2.imread(path)
    from_opencv = img is not None
    if from_opencv:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    else:
        try:
            img = Image.open(path).convert('RGB')
            img = np.array(img)
        except Exception as e:
            print(f"Skipping {path}: {e}")
            return None
    img = cv2.resize(img, (size, size))
    return img.astype(np.float32)

def load_images(paths, size=224):
    images = []
    for p in paths:
        img = load_image(p, size)
        if img is not None:
            images.append(img)
    if not images:
        raise ValueError("No images loaded.")
    return np.stack(images)

def extract_cnn_features(images, model, device):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])
    features = []
    with torch.no_grad():
        for img in images:
            img_tensor = transform(img).unsqueeze(0).to(device)
            feat = model(img_tensor).flatten().cpu().numpy()
            features.append(feat)
    return np.array(features)

def extract_vit_features(images, model, processor, device):
    features = []
    with torch.no_grad():
        for img in images:
            inputs = processor(images=img.astype(np.uint8), return_tensors="pt").to(device)
            outputs = model(**inputs)
            feat = outputs.last_hidden_state[:, 0, :].cpu().numpy().flatten()
            features.append(feat)
    return np.array(features)

def analyze_space(X, name):
    Xc = X - X.mean(axis=0)
    pca = PCA()
    pca.fit(Xc)
    cumsum = np.cumsum(pca.explained_variance_ratio_)
    k90 = np.argmax(cumsum >= 0.90) + 1
    k95 = np.argmax(cumsum >= 0.95) + 1
    k99 = np.argmax(cumsum >= 0.99) + 1
    print(f"{name:12s}: 90%→{k90:2d} PCs, 95%→{k95:2d} PCs, 99%→{k99:2d} PCs")
    pca2 = PCA(n_components=2)
    Z = pca2.fit_transform(Xc)
    return cumsum, pca2.explained_variance_ratio_, Z

def main():
    paths = [
        "/content/test.webp",
        "/content/test2.jpg",
        "/content/test3.jpg",
        "/content/test_4.jpeg"
    ]

    images = load_images(paths, size=224)
    n = len(images)
    print(f"Loaded {n} images\n")

    pixel_flat = np.array([img.flatten() for img in images])
    print(f"Pixel space: {pixel_flat.shape}")

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    cnn_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    cnn_model = torch.nn.Sequential(*list(cnn_model.children())[:-1])
    cnn_model.eval().to(device)
    cnn_features = extract_cnn_features(images, cnn_model, device)
    print(f"CNN features: {cnn_features.shape}")

    try:
        from transformers import ViTModel, ViTImageProcessor
        vit_model = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k')
        vit_processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224-in21k')
        vit_model.eval().to(device)
        vit_features = extract_vit_features(images, vit_model, vit_processor, device)
        print(f"ViT features: {vit_features.shape}")
        has_vit = True
    except ImportError:
        print("Transformers not installed. Skipping ViT.")
        has_vit = False
    except Exception as e:
        print(f"ViT error: {e}. Skipping.")
        has_vit = False

    print("\n--- Dimensionality analysis ---")
    cumsum_pixel, var_pixel, Z_pixel = analyze_space(pixel_flat, "Pixel")
    cumsum_cnn, var_cnn, Z_cnn = analyze_space(cnn_features, "CNN")
    if has_vit:
        cumsum_vit, var_vit, Z_vit = analyze_space(vit_features, "ViT")

    plt.figure(figsize=(6, 4))
    plt.plot(np.arange(1, len(cumsum_pixel)+1), cumsum_pixel, 'o-', label='Pixel Space', color='black')
    plt.plot(np.arange(1, len(cumsum_cnn)+1), cumsum_cnn, 'o-', label='ResNet18', color='blue')
    if has_vit:
        plt.plot(np.arange(1, len(cumsum_vit)+1), cumsum_vit, 'o-', label='ViT', color='green')
    plt.axhline(y=0.90, color='gray', linestyle='--', alpha=0.5)
    plt.axhline(y=0.95, color='gray', linestyle='--', alpha=0.5)
    plt.axhline(y=0.99, color='gray', linestyle='--', alpha=0.5)
    plt.xlabel('Principal components')
    plt.ylabel('Cumulative explained variance')
    plt.title('Feature space comparison')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    fig, axes = plt.subplots(1, 3 if has_vit else 2, figsize=(12, 4))
    axes[0].scatter(Z_pixel[:,0], Z_pixel[:,1], s=50, edgecolors='black')
    for i in range(n):
        axes[0].annotate(str(i+1), (Z_pixel[i,0], Z_pixel[i,1]), fontsize=8, ha='center', va='bottom')
    axes[0].set_xlabel(f'PC1 ({var_pixel[0]:.1%})')
    axes[0].set_ylabel(f'PC2 ({var_pixel[1]:.1%})')
    axes[0].set_title('Raw pixel representation')
    axes[0].grid(alpha=0.3)

    axes[1].scatter(Z_cnn[:,0], Z_cnn[:,1], s=50, edgecolors='black')
    for i in range(n):
        axes[1].annotate(str(i+1), (Z_cnn[i,0], Z_cnn[i,1]), fontsize=8, ha='center', va='bottom')
    axes[1].set_xlabel(f'PC1 ({var_cnn[0]:.1%})')
    axes[1].set_ylabel(f'PC2 ({var_cnn[1]:.1%})')
    axes[1].set_title('ResNet18 feature space')
    axes[1].grid(alpha=0.3)

    if has_vit:
        axes[2].scatter(Z_vit[:,0], Z_vit[:,1], s=50, edgecolors='black')
        for i in range(n):
            axes[2].annotate(str(i+1), (Z_vit[i,0], Z_vit[i,1]), fontsize=8, ha='center', va='bottom')
        axes[2].set_xlabel(f'PC1 ({var_vit[0]:.1%})')
        axes[2].set_ylabel(f'PC2 ({var_vit[1]:.1%})')
        axes[2].set_title('ViT feature space')
        axes[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    print("\n=== Comparison summary ===")
    print(f"{'Representation':<15} {'Dim':>8} {'90%':>6} {'95%':>6} {'99%':>6}")
    print("-" * 45)
    print(f"{'Pixel Space':<15} {pixel_flat.shape[1]:>8} {np.argmax(np.cumsum(cumsum_pixel)>=0.90)+1:>6} {np.argmax(np.cumsum(cumsum_pixel)>=0.95)+1:>6} {np.argmax(np.cumsum(cumsum_pixel)>=0.99)+1:>6}")
    print(f"{'ResNet18':<15} {cnn_features.shape[1]:>8} {np.argmax(np.cumsum(cumsum_cnn)>=0.90)+1:>6} {np.argmax(np.cumsum(cumsum_cnn)>=0.95)+1:>6} {np.argmax(np.cumsum(cumsum_cnn)>=0.99)+1:>6}")
    if has_vit:
        print(f"{'ViT':<15} {vit_features.shape[1]:>8} {np.argmax(np.cumsum(cumsum_vit)>=0.90)+1:>6} {np.argmax(np.cumsum(cumsum_vit)>=0.95)+1:>6} {np.argmax(np.cumsum(cumsum_vit)>=0.99)+1:>6}")

    print("\nDeep feature embeddings are typically more compact than raw pixels,")
    print("requiring fewer principal components to capture the same amount of variance.")

if __name__ == "__main__":
    main()

# Identifying Informative Principal Components

PCA projects each image onto a set of principal components, producing a corresponding set of principal component scores.

For a centered image matrix

$$
X \in \mathbb{R}^{n \times d},
$$

the PCA projection is

$$
Z \in \mathbb{R}^{n \times p},
$$

where each column of $Z$ contains the scores of one principal component for all images.


## Interpretation

The principal component scores can be compared across different image classes to determine whether a component captures meaningful class-related variation.

- Components with similar score distributions contribute little to distinguishing the classes.
- Components with significantly different score distributions indicate directions that separate the image classes.
- These components represent the dominant variations associated with the differences between the groups.


## Insight

Although PCA is an unsupervised technique, the resulting principal component scores can be combined with statistical testing to identify the components that best distinguish different image classes.

This approach provides an interpretable way to investigate which dominant sources of variation contribute most to the separation of the image classes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
from sklearn.decomposition import PCA

def load_images(paths, size=64):
    images = []
    for p in paths:
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            img = cv2.resize(img, (size, size))
            images.append(img.flatten().astype(np.float64))
    if not images:
        raise ValueError("No valid images.")
    return np.stack(images)

def main():
    paths = ["/content/test.webp", "/content/test2.jpg", "/content/test3.jpg", "/content/test_4.jpeg"]
    X = load_images(paths, size=64)

    Xc = X - X.mean(axis=0)

    pca_full = PCA()
    pca_full.fit(Xc)
    cumsum = np.cumsum(pca_full.explained_variance_ratio_)
    n_comp = np.argmax(cumsum >= 0.95) + 1

    pca = PCA(n_components=n_comp)
    scores = pca.fit_transform(Xc)

    print("Explained variance per principal component:")
    for i, ev in enumerate(pca.explained_variance_ratio_):
        print(f"  PC{i+1}: {ev:.2%}")

    print(f"\nMost informative PC: PC1 ({pca.explained_variance_ratio_[0]:.2%} of variance)")

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].bar(np.arange(1, n_comp+1), pca.explained_variance_ratio_, color='skyblue')
    axes[0].set_xlabel('Principal Component')
    axes[0].set_ylabel('Explained Variance Ratio')
    axes[0].set_title('Scree plot')
    axes[0].grid(alpha=0.3)

    eigenimg = pca.components_[0].reshape(64, 64)
    vmax = np.max(np.abs(eigenimg))
    axes[1].imshow(eigenimg, cmap='RdBu', vmin=-vmax, vmax=vmax)
    axes[1].set_title('PC1 eigenimage (most informative)')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
from sklearn.decomposition import PCA
import torch
from torchvision import transforms, models
from mpl_toolkits.mplot3d import Axes3D

class PCADashboard:
    def __init__(self, image_paths, size=64, use_embeddings=False):
        self.image_paths = image_paths
        self.size = size
        self.use_embeddings = use_embeddings
        self.images = None
        self.X = None
        self.pca = None
        self.scores = None
        self.explained_var = None
        self.cumsum = None
        self.components = None
        self.mean_img = None
        self.outlier_threshold = 2.0
        self._load_data()
        self._compute_pca()

    def _load_data(self):
        images = []
        for p in self.image_paths:
            img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                img = cv2.resize(img, (self.size, self.size))
                images.append(img.flatten().astype(np.float64))
        if not images:
            raise ValueError("No valid images.")
        self.images = np.stack(images)
        if self.use_embeddings:
            self._extract_embeddings()
        else:
            self.X = self.images

    def _extract_embeddings(self):
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        model = torch.nn.Sequential(*list(model.children())[:-1])
        model.eval().to(device)
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225])
        ])
        imgs_rgb = []
        for img in self.images:
            img_rgb = cv2.cvtColor(img.reshape(self.size, self.size).astype(np.uint8), cv2.COLOR_GRAY2RGB)
            img_rgb = cv2.resize(img_rgb, (224, 224)).astype(np.float32)
            imgs_rgb.append(img_rgb)
        embeddings = []
        with torch.no_grad():
            for img in imgs_rgb:
                img_tensor = transform(img).unsqueeze(0).to(device)
                emb = model(img_tensor).flatten().cpu().numpy()
                embeddings.append(emb)
        self.X = np.stack(embeddings)

    def _compute_pca(self):
        Xc = self.X - self.X.mean(axis=0)
        self.pca = PCA()
        self.pca.fit(Xc)
        self.scores = self.pca.transform(Xc)
        self.explained_var = self.pca.explained_variance_ratio_
        self.cumsum = np.cumsum(self.explained_var)
        self.components = self.pca.components_
        self.mean_img = self.X.mean(axis=0)

    def plot_overview(self):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
        ax1.bar(np.arange(1, len(self.explained_var)+1), self.explained_var, color='skyblue', edgecolor='black')
        ax1.set_xlabel('Principal Component')
        ax1.set_ylabel('Explained Variance')
        ax1.set_title('Explained Variance')
        ax1.grid(alpha=0.3)
        ax2.plot(np.arange(1, len(self.cumsum)+1), self.cumsum, 'o-', color='black')
        ax2.axhline(y=0.95, color='red', linestyle='--', label='95%')
        ax2.axhline(y=0.99, color='blue', linestyle='--', label='99%')
        ax2.set_xlabel('Number of Components')
        ax2.set_ylabel('Cumulative Explained Variance')
        ax2.set_title('Cumulative Variance')
        ax2.legend()
        ax2.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()
        print(f"Images: {len(self.images)}, Dimension: {self.X.shape[1]}")
        print(f"Mode: {'Embedding' if self.use_embeddings else 'Pixel'} Space")
        print(f"95% variance: {np.argmax(self.cumsum >= 0.95) + 1} components")

    def plot_2d(self, highlight_outliers=False):
        scores_2d = self.scores[:, :2]
        plt.figure(figsize=(8, 6))
        colors = plt.cm.viridis(np.linspace(0, 1, len(self.images)))
        plt.scatter(scores_2d[:, 0], scores_2d[:, 1], c=colors, s=50, edgecolors='black')
        for i in range(len(self.images)):
            plt.annotate(str(i+1), (scores_2d[i, 0], scores_2d[i, 1]), fontsize=7, ha='center', va='bottom')
        if highlight_outliers:
            center = scores_2d.mean(axis=0)
            dist = np.linalg.norm(scores_2d - center, axis=1)
            threshold = dist.mean() + self.outlier_threshold * dist.std()
            outliers = np.where(dist > threshold)[0]
            if len(outliers) > 0:
                plt.scatter(scores_2d[outliers, 0], scores_2d[outliers, 1], s=120, facecolors='none', edgecolors='red', linewidths=2, label='Outliers')
                plt.legend()
        plt.xlabel(f'PC1 ({self.explained_var[0]:.1%})')
        plt.ylabel(f'PC2 ({self.explained_var[1]:.1%})')
        plt.title('2D PCA Projection')
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

    def plot_3d(self):
        scores_3d = self.scores[:, :3]
        fig = plt.figure(figsize=(8, 6))
        ax = fig.add_subplot(111, projection='3d')
        colors = plt.cm.viridis(np.linspace(0, 1, len(self.images)))
        ax.scatter(scores_3d[:, 0], scores_3d[:, 1], scores_3d[:, 2], c=colors, s=40, edgecolors='black')
        for i in range(len(self.images)):
            ax.text(scores_3d[i, 0], scores_3d[i, 1], scores_3d[i, 2], str(i+1), fontsize=7)
        ax.set_xlabel(f'PC1 ({self.explained_var[0]:.1%})')
        ax.set_ylabel(f'PC2 ({self.explained_var[1]:.1%})')
        ax.set_zlabel(f'PC3 ({self.explained_var[2]:.1%})')
        ax.set_title('3D PCA Projection')
        plt.tight_layout()
        plt.show()

    def plot_basis(self, n=4):
        if self.use_embeddings:
            print("Basis images only available in Pixel Space.")
            return
        h = w = self.size
        n = min(n, len(self.components))
        fig, axes = plt.subplots(1, n, figsize=(3*n, 3))
        if n == 1:
            axes = [axes]
        for i in range(n):
            basis = self.components[i].reshape(h, w)
            vmax = np.max(np.abs(basis))
            axes[i].imshow(basis, cmap='RdBu', vmin=-vmax, vmax=vmax)
            axes[i].set_title(f'PC{i+1}')
            axes[i].axis('off')
        plt.suptitle('Principal Components (Basis Images)')
        plt.tight_layout()
        plt.show()

    def plot_reconstruction(self, k=None, idx=0):
        if self.use_embeddings:
            print("Reconstruction only available in Pixel Space.")
            return
        if k is None:
            k = min(2, len(self.components))
        h = w = self.size
        Xc = self.images - self.images.mean(axis=0)
        V = self.components[:k, :].T
        Z = Xc @ V
        X_recon = Z @ V.T + self.images.mean(axis=0)
        orig = self.images[idx].reshape(h, w)
        recon = X_recon[idx].reshape(h, w)
        residual = orig - recon
        fig, axes = plt.subplots(1, 3, figsize=(9, 3))
        axes[0].imshow(orig, cmap='gray')
        axes[0].set_title('Original')
        axes[0].axis('off')
        axes[1].imshow(np.clip(recon, 0, 255), cmap='gray')
        axes[1].set_title(f'Reconstructed (k={k})')
        axes[1].axis('off')
        vmax = np.max(np.abs(residual))
        axes[2].imshow(residual, cmap='RdBu', vmin=-vmax, vmax=vmax)
        axes[2].set_title('Residual')
        axes[2].axis('off')
        plt.suptitle('PCA Reconstruction')
        plt.tight_layout()
        plt.show()
        mse = np.mean((orig - recon)**2)
        psnr = 20 * np.log10(255.0 / (np.sqrt(mse) + 1e-8))
        print(f"k={k}, MSE={mse:.2f}, PSNR={psnr:.2f}dB")

    def run(self):
        self.plot_overview()
        self.plot_2d(highlight_outliers=True)
        self.plot_3d()
        if not self.use_embeddings:
            self.plot_basis()
            self.plot_reconstruction(k=2)

if __name__ == "__main__":
    paths = [
        "/content/test.webp",
        "/content/test2.jpg",
        "/content/test3.jpg",
        "/content/test_4.jpeg"
    ]
    dashboard = PCADashboard(paths, size=64)
    dashboard.run()